<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 5장 연습 문제 해답(Chapter 5 Exercise solutions)

In [ ]:
from importlib.metadata import version

pkgs = ["numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # OpenAI의 사전학습된 가중치를 위해
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

# 연습 문제 5.1: 온도 스케일된 소프트맥스 점수와 샘플링 확률(Temperature-scaled softmax scores and sampling probabilities)

- 이 섹션에서 정의한 `print_sampled_tokens` 함수를 사용하여 "pizza"라는 단어가 샘플링되는 횟수를 출력할 수 있습니다
- 섹션 5.3.1에서 정의한 코드부터 시작해보겠습니다

- 온도가 0이나 0.1일 때는 0번 샘플링되고, 온도가 5로 스케일업될 때는 32번 샘플링됩니다. 추정 확률은 32/1000 * 100% = 3.2%입니다

- 실제 확률은 4.3%이며 재스케일된 소프트맥스 확률 텐서(`scaled_probas[2][6]`)에 포함되어 있습니다

- 아래는 5장의 코드를 사용한 독립적인 예제입니다:

In [ ]:
import torch

vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 
inverse_vocab = {v: k for k, v in vocab.items()}

next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

def print_sampled_tokens(probas):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")


def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


temperatures = [1, 0.1, 5]  # 원래, 더 높은, 더 낮은 온도
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

- 이제 `scaled_probas`를 반복하여 각 경우의 샘플링 빈도를 출력할 수 있습니다:

In [ ]:
for i, probas in enumerate(scaled_probas):
    print("\n\n온도:", temperatures[i])
    print_sampled_tokens(probas)

- 샘플링은 "pizza"라는 단어가 샘플링될 때 실제 확률의 근사치를 제공한다는 점에 주목하세요
- 예를 들어, 1000번 중 32번 샘플링된다면 추정 확률은 3.2%입니다
- 실제 확률을 얻으려면 `scaled_probas`의 해당 항목을 확인하여 확률을 직접 확인할 수 있습니다

- "pizza"는 어휘에서 7번째 항목이므로, 온도 5에 대해서는 다음과 같이 얻습니다:

In [ ]:
temp5_idx = 2
pizza_idx = 6

scaled_probas[temp5_idx][pizza_idx]

온도가 5로 설정된 경우 "pizza"라는 단어가 샘플링될 확률은 4.3%입니다

# 연습 문제 5.2: 다양한 온도 및 top-k 설정(Different temperature and top-k settings)

- 온도와 top-k 설정 모두 개별 LLM에 따라 조정되어야 합니다(원하는 출력을 생성할 때까지 시행착오 과정)
- 원하는 결과는 응용 프로그램에 따라서도 다릅니다
  - 낮은 top-k와 온도는 덜 무작위적인 결과를 가져오며, 이는 교육 콘텐츠, 기술 문서 작성, 질문 답변, 데이터 분석, 코드 생성 등에서 바람직합니다
  - 높은 top-k와 온도는 더 다양하고 무작위적인 출력을 가져오며, 이는 브레인스토밍 작업, 창의적 글쓰기 등에서 더 바람직합니다

# 연습 문제 5.3: 디코딩 함수의 결정론적 동작(Deterministic behavior in the decoding functions)

`generate` 함수에서 결정론적 동작을 강제하는 방법은 여러 가지가 있습니다:

1. `temperature=0.0`으로 설정;
2. `top_k=1`로 설정.

아래는 5장의 코드를 사용한 독립적인 예제입니다:

In [ ]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,  # 어휘 크기
    "context_length": 256,       # 축소된 컨텍스트 길이 (원래: 1024)
    "emb_dim": 768,       # 임베딩 차원
    "n_heads": 12,        # 어텐션 헤드 수
    "n_layers": 12,       # 레이어 수
    "drop_rate": 0.1,     # 드롭아웃 비율
    "qkv_bias": False     # Query-key-value 바이어스
}


torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth", weights_only=True))
model.eval();

In [ ]:
from gpt_generate import generate, text_to_token_ids, token_ids_to_text
from previous_chapters import generate_text_simple

In [ ]:
# torch.argmax를 사용한 결정론적 함수

start_context = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
# 결정론적 동작: top_k 없음, 온도 스케일링 없음

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=None,
    temperature=0.0
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))

- 이전 코드 셀을 다시 실행하면 정확히 동일한 생성된 텍스트가 생성됩니다:

In [ ]:
# 결정론적 동작: top_k 없음, 온도 스케일링 없음

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=None,
    temperature=0.0
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))

# 연습 문제 5.4: 지속적인 사전학습(Continued pretraining)

- 5장에서 모델을 처음 훈련한 Python 세션에 아직 있다면, 한 에폭을 더 계속 사전학습하려면 메인 장에서 저장한 모델과 옵티마이저를 로드하고 `train_model_simple` 함수를 다시 호출하기만 하면 됩니다

- 이 새로운 코드 환경에서 재현 가능하게 만들려면 몇 단계가 더 필요합니다
- 먼저 토크나이저, 모델, 옵티마이저를 로드합니다:

In [ ]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 어휘 크기
    "context_length": 256, # 축소된 컨텍스트 길이 (원래: 1024)
    "emb_dim": 768,        # 임베딩 차원
    "n_heads": 12,         # 어텐션 헤드 수
    "n_layers": 12,        # 레이어 수
    "drop_rate": 0.1,      # 드롭아웃 비율
    "qkv_bias": False      # Query-key-value 바이어스
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = tiktoken.get_encoding("gpt2")

checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

- 다음으로, 데이터 로더를 초기화합니다:

In [ ]:
import os
import urllib.request
from previous_chapters import create_dataloader_v1


file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


# 훈련/검증 비율
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

- 마지막으로, `train_model_simple` 함수를 사용하여 모델을 훈련합니다:

In [ ]:
from gpt_train import train_model_simple

num_epochs = 1
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# 연습 문제 5.5: 사전학습된 모델의 훈련 및 검증 세트 손실(Training and validation set losses of the pretrained model)

- GPT 모델의 훈련 및 검증 세트 손실을 계산하기 위해 다음 코드를 사용할 수 있습니다:

```python
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)
```

- 124M 매개변수에 대한 결과 손실은 다음과 같습니다:

```
Training loss: 3.754748503367106
Validation loss: 3.559617757797241
```

- 주요 관찰은 훈련 및 검증 세트 성능이 비슷한 범위에 있다는 것입니다
- 이는 여러 가지 설명이 있을 수 있습니다:

1. The Verdict는 OpenAI가 GPT-2를 훈련할 때 사전학습 데이터셋의 일부가 아니었습니다. 따라서 모델이 명시적으로 훈련 세트에 과적합하지 않으며 The Verdict의 훈련 및 검증 세트 부분에서 유사하게 잘 수행됩니다. (검증 세트 손실이 훈련 세트 손실보다 약간 낮은데, 이는 딥러닝에서 비정상적입니다. 그러나 데이터셋이 상대적으로 작기 때문에 무작위 노이즈 때문일 가능성이 높습니다. 실제로 과적합이 없다면 훈련 및 검증 세트 성능이 대략 동일할 것으로 예상됩니다).

2. The Verdict가 GPT-2의 훈련 데이터셋의 일부였습니다. 이 경우, 검증 세트도 훈련에 사용되었을 것이므로 모델이 훈련 데이터를 과적합하는 정도를 말할 수 없습니다. 과적합 정도를 평가하려면 OpenAI가 GPT-2 훈련을 완료한 후에 생성된 새로운 데이터셋이 필요하여 사전학습에 포함될 수 없었던 것을 확실히 해야 합니다.

아래 코드는 이 새로운 노트북에 대한 재현 가능한 독립적인 예제입니다.

In [ ]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 어휘 크기
    "context_length": 256, # 축소된 컨텍스트 길이 (원래: 1024)
    "emb_dim": 768,        # 임베딩 차원
    "n_heads": 12,         # 어텐션 헤드 수
    "n_layers": 12,        # 레이어 수
    "drop_rate": 0.1,      # 드롭아웃 비율
    "qkv_bias": False      # Query-key-value 바이어스
}


torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
# 간결성을 위해 딕셔너리에 모델 구성 정의
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# 기본 구성을 복사하고 특정 모델 설정으로 업데이트
model_name = "gpt2-small (124M)"  # 예제 모델 이름
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [ ]:
from gpt_generate import load_weights_into_gpt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_weights_into_gpt(gpt, params)
gpt.to(device);

In [ ]:
import os
import urllib.request
from previous_chapters import create_dataloader_v1


file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


# 훈련/검증 비율
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
from gpt_train import calc_loss_loader

torch.manual_seed(123) # 데이터 로더의 셔플링으로 인한 재현가능성을 위해
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)

print("훈련 손실:", train_loss)
print("검증 손실:", val_loss)

가장 큰 GPT-2 모델에 대해서도 이를 반복할 수 있지만, 컨텍스트 길이를 업데이트하는 것을 잊지 마세요:

In [ ]:
settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")

model_name = "gpt2-xl (1558M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

load_weights_into_gpt(gpt, params)
gpt.to(device)

torch.manual_seed(123)
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)

print("훈련 손실:", train_loss)
print("검증 손실:", val_loss)

# 연습 문제 5.6: 더 큰 모델 시도하기(Trying larger models)

- 메인 장에서는 124M 매개변수만 가진 가장 작은 GPT-2 모델을 실험했습니다
- 이유는 자원 요구사항을 가능한 한 낮게 유지하기 위함이었습니다
- 그러나 최소한의 코드 변경으로 더 큰 모델을 쉽게 실험할 수 있습니다
- 예를 들어, 5장에서 124M 대신 1558M 모델을 로드하려면 변경해야 하는 코드는 다음 2줄뿐입니다

```python
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")
model_name = "gpt2-small (124M)"
```

- 업데이트된 코드는 다음과 같습니다


```python
settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")
model_name = "gpt2-xl (1558M)"
```

In [ ]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 어휘 크기
    "context_length": 256, # 축소된 컨텍스트 길이 (원래: 1024)
    "emb_dim": 768,        # 임베딩 차원
    "n_heads": 12,         # 어텐션 헤드 수
    "n_layers": 12,        # 레이어 수
    "drop_rate": 0.1,      # 드롭아웃 비율
    "qkv_bias": False      # Query-key-value 바이어스
}


tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
from gpt_download import download_and_load_gpt2
from gpt_generate import load_weights_into_gpt


model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-xl (1558M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")
load_weights_into_gpt(gpt, params)

In [ ]:
from gpt_generate import generate, text_to_token_ids, token_ids_to_text

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))